In [14]:
import sys

if 'workhorse' not in sys.executable.split('/'):
    origin = 'workspace/'
    sys.path.append('/media/')
else:
    origin = 'data/Aldhani/eoagritwin/'
    sys.path.append('/home/potzschf/repos/')

from helperToolz.polygons_to_labels import *
from helperToolz.helpsters import *
import xarray as xr
import rioxarray
from helperToolz.feevos.rocksdbutils_copy import *
from math import ceil as mceil
variables2use=['B2','B3','B4','B8']

In [ ]:
arr = stackReader(gdal.Open('/workspace/fields/Fine_tune/label/BRB/2022/BRB_2022_rowstart_4096_colstart_17664.tif'))

In [15]:
nc_file = xr.open_dataset('/workspace/fields/Fine_tune/img/BRB/2022/BRB_2022_rowstart_4096_colstart_17664.nc')
image = np.concatenate([nc_file[var].values[None] for var in variables2use],0)

In [ ]:
# load a rocksdb dataset
fine_set = RocksDBDataset('/workspace/fields/output/rocks_db/IACS_as_AI4_for_model_from_scratch_dilate_false.db/valid.db', transform=None)

In [ ]:
ai4_set = RocksDBDataset('/workspace/fields/output/rocks_db/AI4_RGB_exclude_True.db/train.db', transform=None)

In [ ]:
ai4_data = []
ai4_labels = []
for i in range(len(ai4_set)):
    ai4_data.append(np.unique(ai4_set[i][0]))
    ai4_labels.append(np.unique(ai4_set[i][1]))

In [ ]:
fine_data = []
fine_labels = []
for i in range(len(fine_set)):
    fine_data.append(np.unique(fine_set[i][0], return_counts=True))
    fine_labels.append(np.unique(fine_set[i][1]))

In [ ]:
for i in range(4):
    for j in range(6):
        plotter(fine_set[150][0][i,j,:,:])

In [ ]:
plotter(fine_set[150][1][0,:,:])


In [ ]:
plotter(fine_set[0][0][0,0,:,:])

In [ ]:
no_count = [f for fd in fine_data for f in fd[0]]


In [ ]:
for count, i in enumerate(fine_data):
    if np.isnan(i[0]).any():
        print(count)

In [ ]:
ai4_data_uni = np.unique([f for fd in ai4_data for f in fd])
ai4_labels_uni = np.unique([f for fd in ai4_labels for f in fd])

In [ ]:
# check nc files
variables2use=['B2','B3','B4','B8']
nc_files = getFilelist(f'/{origin}fields/Fine_tune/', '.nc', deep=True)

In [ ]:
for i in range(len(nc_files)):
    nc_file = xr.open_dataset(nc_files[i])
    image = np.concatenate([nc_file[var].values[None] for var in variables2use],0)
    if len(np.unique(image)) == 1:
        print(i)
        print(nc_files[i])
        print(np.unique(image))

In [ ]:
label_files = getFilelist(f'/{origin}fields/Fine_tune/', '.tif', deep=True)
for i in range(len(label_files)):
    arr = stackReader(label_files[i])
    print(np.unique(arr))

In [ ]:
# mock the normalisation when pressing into rocks
normi = AI4BNormal_S2()

for i in range(len(nc_files)):
    nc_file = xr.open_dataset(nc_files[i])
    image = np.concatenate([nc_file[var].values[None] for var in variables2use],0)
    #plotter(image)
    if np.isnan(np.unique(normi(image))).any() == True:
        print(f'{i}: {nc_files[i]}')

In [ ]:
i = 995
nc_file = xr.open_dataset(nc_files[i])
image = np.concatenate([nc_file[var].values[None] for var in variables2use],0)
norm_image = normi(image)
image[np.where(np.isnan(norm_image) == True)]

In [ ]:
np.where(np.isnan(norm_image) == True)